============================================
 
Kitsap Transit — StreetLight Data 
============================================
 
Top Routes | Origin / Destination
 
ORIGIN DESTINATION MATRIX HEATMAP
 

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
import warnings
import sys
import os

In [ ]:
warnings.filterwarnings('ignore')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
CSV_PATH = r"C:\Users\rebeccasc\Documents\Scripts\BI_commerce_TDM\2014404_BI_OD_Top_Routes\2014404_BI_OD_Top_Routes_tr_od.csv"
OUTPUT_DIR = os.path.dirname(os.path.abspath(__file__))
TOP_N = 10  # Number of top routes to show

In [ ]:
# ── Brand palette (Kitsap Transit-adjacent: navy/teal/amber) ──────────────────
NAVY    = "#0D2340"
TEAL    = "#1A6B72"
AMBER   = "#E8A020"
RUST    = "#C04E2E"
SAGE    = "#4A7C6F"
CREAM   = "#F5F0E8"
LGRAY   = "#E8E4DC"
MGRAY   = "#A09888"

In [ ]:
ZONE_COLORS = {
    "HIGH_SCHOOL_RD": "#1A6B72",
    "SPORTSMAN_CLUB": "#E8A020",
    "SR305_S":        "#C04E2E",
    "WINSLOW_WAY":    "#4A7C6F",
}

In [ ]:
DAY_ORDER = [
    "0: All Days (M-Su)",
    "1: Monday (M-M)",
    "2: Tuesday (Tu-Tu)",
    "3: Wednesday (W-W)",
    "4: Thursday (Th-Th)",
    "5: Friday (F-F)",
    "6: Saturday (Sa-Sa)",
    "7: Sunday (Su-Su)",
]

In [ ]:
DAY_SHORT = {
    "0: All Days (M-Su)": "All Days",
    "1: Monday (M-M)":    "Mon",
    "2: Tuesday (Tu-Tu)": "Tue",
    "3: Wednesday (W-W)": "Wed",
    "4: Thursday (Th-Th)":"Thu",
    "5: Friday (F-F)":    "Fri",
    "6: Saturday (Sa-Sa)":"Sat",
    "7: Sunday (Su-Su)":  "Sun",
}

In [ ]:
HOUR_ORDER = [f"{h:02d}" for h in range(25)]  # 00-24

═══════════════════════════════════════════════════════════════════════════════
1. LOAD & CLEAN
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def load_data(path):
    print(f"Loading: {path}")
    try:
        df = pd.read_csv(path, sep="\t", encoding="utf-8", low_memory=False)
    except Exception:
        df = pd.read_csv(path, encoding="utf-8", low_memory=False)

    df.columns = df.columns.str.strip()

    # Rename for convenience
    rename = {
        "Origin Zone Name":                    "origin",
        "Destination Zone Name":               "destination",
        "Segment Name":                        "segment",
        "Segment Type":                        "seg_type",
        "Day Type":                            "day_type",
        "Day Part":                            "day_part",
        "Average Daily O-D Traffic (StL Volume)": "od_vol",
        "Average Daily O-S-D Traffic (StL Volume)": "osd_vol",
        "Average Daily Origin Zone Traffic (StL Volume)": "origin_vol",
        "Average Daily Destination Zone Traffic (StL Volume)": "dest_vol",
        "Trip Proportion":                     "trip_prop",
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})

    # Numeric coercion
    for col in ["od_vol", "osd_vol", "origin_vol", "dest_vol", "trip_prop"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # OD pair label
    df["od_pair"] = df["origin"] + " → " + df["destination"]

    # Extract hour code from day_part string e.g. "08: 7am (7am-8am)" → "08"
    if "day_part" in df.columns:
        df["hour_code"] = df["day_part"].str.extract(r"^(\d+):")

    print(f"  Rows: {len(df):,}  |  Columns: {list(df.columns)}")
    return df

═══════════════════════════════════════════════════════════════════════════════
2. SUMMARIES
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def summarize_all_days(df):
    """Aggregate on Day Type=0 (All Days) + Day Part=00 (All Day)."""
    mask = (
        df["day_type"].str.startswith("0:") &
        df["day_part"].str.startswith("00:")
    )
    base = df[mask].copy()

    # Top routes by OD volume
    top = (
        base.groupby(["od_pair", "origin", "destination", "segment", "seg_type"])
        ["od_vol"].sum()
        .reset_index()
        .sort_values("od_vol", ascending=False)
    )
    return top

In [ ]:
def segment_by_od_pair(df):
    """For each OD pair (all days / all day), aggregate segment volumes."""
    mask = (
        df["day_type"].str.startswith("0:") &
        df["day_part"].str.startswith("00:")
    )
    base = df[mask].copy()
    grp = (
        base.groupby(["od_pair", "segment", "seg_type"])["od_vol"]
        .sum().reset_index()
        .sort_values(["od_pair", "od_vol"], ascending=[True, False])
    )
    return grp

In [ ]:
def hourly_od_volumes(df):
    """All Days, hourly breakdown by OD pair."""
    mask = df["day_type"].str.startswith("0:")
    base = df[mask & ~df["day_part"].str.startswith("00:")].copy()
    grp = (
        base.groupby(["od_pair", "day_part", "hour_code"])["od_vol"]
        .sum().reset_index()
    )
    return grp

In [ ]:
def daily_od_volumes(df):
    """All-day (00) volume by day type and OD pair."""
    mask = df["day_part"].str.startswith("00:")
    base = df[mask].copy()
    grp = (
        base.groupby(["od_pair", "day_type"])["od_vol"]
        .sum().reset_index()
    )
    return grp

═══════════════════════════════════════════════════════════════════════════════
3. PLOTTING HELPERS
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def style_ax(ax, title="", xlabel="", ylabel="", grid_axis="y"):
    ax.set_facecolor(CREAM)
    ax.set_title(title, fontsize=11, fontweight="bold", color=NAVY, pad=8)
    ax.set_xlabel(xlabel, fontsize=9, color=NAVY)
    ax.set_ylabel(ylabel, fontsize=9, color=NAVY)
    ax.tick_params(colors=NAVY, labelsize=8)
    for spine in ax.spines.values():
        spine.set_color(LGRAY)
    if grid_axis:
        ax.grid(axis=grid_axis, color=LGRAY, linewidth=0.8, linestyle="--", alpha=0.7)
        ax.set_axisbelow(True)

In [ ]:
def od_color(od_pair):
    for zone, color in ZONE_COLORS.items():
        if od_pair.startswith(zone):
            return color
    return MGRAY

═══════════════════════════════════════════════════════════════════════════════
4. FIGURE 1 – OVERVIEW DASHBOARD
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def fig_overview(top_routes, title_suffix="All Days | All Day"):
    top = top_routes.drop_duplicates("od_pair").head(TOP_N).copy()
    top = top.sort_values("od_vol")

    fig = plt.figure(figsize=(14, 8), facecolor=NAVY)
    fig.patch.set_facecolor(NAVY)

    # Header band
    fig.text(0.5, 0.96, "StreetLight OD Top Routes Analysis", ha="center",
             fontsize=18, fontweight="bold", color=CREAM,
             fontfamily="DejaVu Serif")
    fig.text(0.5, 0.925, f"Analysis 2014404 · Kitsap Transit · Jan–Dec 2025 · {title_suffix}",
             ha="center", fontsize=10, color=MGRAY)

    gs = gridspec.GridSpec(1, 2, figure=fig, left=0.04, right=0.96,
                           top=0.88, bottom=0.08, wspace=0.35)

    # ── Left: Horizontal bar chart ──────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    ax1.set_facecolor(CREAM)

    colors = [od_color(p) for p in top["od_pair"]]
    bars = ax1.barh(range(len(top)), top["od_vol"], color=colors,
                    edgecolor="white", linewidth=0.5, height=0.65)

    ax1.set_yticks(range(len(top)))
    ax1.set_yticklabels(top["od_pair"], fontsize=8.5, color=NAVY)
    ax1.set_xlabel("Avg Daily O-D Volume (StL)", fontsize=9, color=NAVY)
    ax1.set_title(f"Top {len(top)} OD Routes by Volume", fontsize=11,
                  fontweight="bold", color=NAVY, pad=8)

    for i, (bar, val) in enumerate(zip(bars, top["od_vol"])):
        ax1.text(val + max(top["od_vol"]) * 0.01, bar.get_y() + bar.get_height() / 2,
                 f"{int(val):,}", va="center", fontsize=8, color=NAVY, fontweight="bold")

    style_ax(ax1, grid_axis="x")
    ax1.set_facecolor(CREAM)

    # ── Right: Summary table ────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])
    ax2.set_facecolor(CREAM)
    ax2.axis("off")

    display = top.sort_values("od_vol", ascending=False).head(TOP_N).copy()
    display["Rank"] = range(1, len(display) + 1)
    display["OD Pair"] = display["od_pair"]
    display["Avg Daily Vol"] = display["od_vol"].apply(lambda x: f"{int(x):,}")
    display["Top Segment"] = display["segment"].str.split(" / ").str[0].str[:28]
    display["Type"] = display["seg_type"].str[:10]
    show_cols = ["Rank", "OD Pair", "Avg Daily Vol", "Top Segment", "Type"]

    tbl = ax2.table(
        cellText=display[show_cols].values,
        colLabels=show_cols,
        cellLoc="left",
        loc="center",
        bbox=[0, 0, 1, 1],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)

    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor(LGRAY)
        if row == 0:
            cell.set_facecolor(NAVY)
            cell.set_text_props(color=CREAM, fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor(LGRAY)
        else:
            cell.set_facecolor(CREAM)
        cell.set_text_props(color=NAVY) if row > 0 else None

    ax2.set_title("Ranked Route Summary", fontsize=11,
                  fontweight="bold", color=NAVY, pad=8)

    # Legend
    handles = [mpatches.Patch(color=c, label=z) for z, c in ZONE_COLORS.items()]
    fig.legend(handles=handles, title="Origin Zone", loc="lower center",
               ncol=4, fontsize=8, title_fontsize=8,
               framealpha=0.15, facecolor=CREAM, edgecolor=MGRAY,
               bbox_to_anchor=(0.5, 0.01))

    out = os.path.join(OUTPUT_DIR, "fig1_overview.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=NAVY)
    plt.close()
    print(f"  ✓ Saved: {out}")

═══════════════════════════════════════════════════════════════════════════════
5. FIGURE 2 – HOURLY HEATMAP
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def fig_hourly_heatmap(df, n_pairs=8):
    hourly = hourly_od_volumes(df)

    # Pick top OD pairs by total volume
    top_pairs = (
        hourly.groupby("od_pair")["od_vol"].sum()
        .nlargest(n_pairs).index.tolist()
    )
    hourly = hourly[hourly["od_pair"].isin(top_pairs)]

    pivot = hourly.pivot_table(index="od_pair", columns="hour_code",
                               values="od_vol", aggfunc="sum", fill_value=0)

    # Sort columns numerically
    pivot = pivot.reindex(
        columns=sorted(pivot.columns, key=lambda x: int(x)), fill_value=0
    )
    # Sort rows by total
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    cmap = LinearSegmentedColormap.from_list(
        "kt", [CREAM, TEAL, AMBER, RUST], N=256
    )

    fig, ax = plt.subplots(figsize=(16, 5), facecolor=NAVY)
    fig.patch.set_facecolor(NAVY)
    ax.set_facecolor(NAVY)

    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap, interpolation="nearest")

    # Labels
    hour_labels = []
    for hc in pivot.columns:
        h = int(hc)
        label = f"{h}am" if h < 12 else ("12pm" if h == 12 else f"{h-12}pm")
        if h == 0: label = "12am"
        if h == 24: label = "12am+"
        hour_labels.append(label)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(hour_labels, fontsize=7.5, color=CREAM, rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9, color=CREAM)

    # Annotate cells with values where notable
    max_val = pivot.values.max()
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if v > max_val * 0.3:
                ax.text(j, i, str(int(v)), ha="center", va="center",
                        fontsize=6.5, color="white" if v > max_val * 0.6 else NAVY,
                        fontweight="bold")

    cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.01)
    cbar.ax.tick_params(colors=CREAM, labelsize=8)
    cbar.set_label("Avg Daily Volume", color=CREAM, fontsize=9)

    ax.set_title("Hourly O-D Volume Heatmap  ·  All Days  ·  Top Route Pairs",
                 fontsize=13, fontweight="bold", color=CREAM, pad=10,
                 fontfamily="DejaVu Serif")
    ax.set_xlabel("Hour of Day", fontsize=9, color=CREAM)

    plt.tight_layout(pad=1.5)
    out = os.path.join(OUTPUT_DIR, "fig2_hourly_heatmap.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=NAVY)
    plt.close()
    print(f"  ✓ Saved: {out}")

═══════════════════════════════════════════════════════════════════════════════
6. FIGURE 3 – DAY-OF-WEEK COMPARISON
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def fig_day_of_week(df, n_pairs=6):
    daily = daily_od_volumes(df)

    top_pairs = (
        daily[daily["day_type"].str.startswith("0:")]
        .groupby("od_pair")["od_vol"].sum()
        .nlargest(n_pairs).index.tolist()
    )

    daily = daily[
        daily["od_pair"].isin(top_pairs) &
        ~daily["day_type"].str.startswith("0:")  # exclude "All Days" aggregate
    ].copy()
    daily["day_short"] = daily["day_type"].map(DAY_SHORT)

    # Sort days
    day_order_short = [DAY_SHORT[d] for d in DAY_ORDER if d in DAY_SHORT and not d.startswith("0:")]

    fig, axes = plt.subplots(2, 3, figsize=(15, 7), facecolor=NAVY)
    fig.patch.set_facecolor(NAVY)
    fig.suptitle("Average Daily O-D Volume by Day of Week  ·  Top Route Pairs",
                 fontsize=14, fontweight="bold", color=CREAM,
                 fontfamily="DejaVu Serif", y=1.01)

    colors_cycle = [TEAL, AMBER, RUST, SAGE, "#7B5EA7", "#C0753A"]

    for ax, (pair, color) in zip(axes.flat, zip(top_pairs, colors_cycle)):
        sub = daily[daily["od_pair"] == pair].copy()
        sub["day_short"] = pd.Categorical(sub["day_short"], categories=day_order_short, ordered=True)
        sub = sub.sort_values("day_short")

        ax.set_facecolor(CREAM)
        bars = ax.bar(sub["day_short"], sub["od_vol"],
                      color=color, edgecolor="white", linewidth=0.5, width=0.65)

        for bar, val in zip(bars, sub["od_vol"]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(sub["od_vol"]) * 0.02,
                    f"{int(val)}", ha="center", va="bottom", fontsize=7.5,
                    color=NAVY, fontweight="bold")

        style_ax(ax, title=pair, ylabel="Avg Daily Volume")
        ax.tick_params(axis="x", rotation=30)

    # Hide any empty subplots
    for ax in axes.flat[len(top_pairs):]:
        ax.set_visible(False)

    plt.tight_layout(pad=1.5)
    out = os.path.join(OUTPUT_DIR, "fig3_day_of_week.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=NAVY)
    plt.close()
    print(f"  ✓ Saved: {out}")

═══════════════════════════════════════════════════════════════════════════════
7. FIGURE 4 – TOP SEGMENTS PER OD PAIR
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def fig_top_segments(df, n_od=4, n_segs=5):
    segs = segment_by_od_pair(df)

    top_od_pairs = (
        segs.groupby("od_pair")["od_vol"].sum()
        .nlargest(n_od).index.tolist()
    )

    fig, axes = plt.subplots(1, n_od, figsize=(15, 5.5), facecolor=NAVY)
    fig.patch.set_facecolor(NAVY)
    fig.suptitle("Top Street Segments Used per OD Pair  ·  All Days | All Day",
                 fontsize=13, fontweight="bold", color=CREAM,
                 fontfamily="DejaVu Serif", y=1.01)

    seg_type_colors = {
        "Primary": RUST,
        "Secondary": TEAL,
        "Tertiary": SAGE,
        "Residential": AMBER,
        "motorway": NAVY,
    }

    for ax, pair in zip(axes, top_od_pairs):
        sub = segs[segs["od_pair"] == pair].head(n_segs).copy()
        sub["seg_short"] = sub["segment"].str.split(" / ").str[0].str[:25]
        sub = sub.sort_values("od_vol")

        colors = [seg_type_colors.get(t, MGRAY) for t in sub["seg_type"]]
        ax.set_facecolor(CREAM)
        bars = ax.barh(range(len(sub)), sub["od_vol"], color=colors,
                       edgecolor="white", linewidth=0.5, height=0.65)
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(sub["seg_short"], fontsize=8, color=NAVY)

        for bar, val in zip(bars, sub["od_vol"]):
            ax.text(val + max(sub["od_vol"]) * 0.02,
                    bar.get_y() + bar.get_height() / 2,
                    f"{int(val):,}", va="center", fontsize=7.5,
                    color=NAVY, fontweight="bold")

        style_ax(ax, title=pair[:35], xlabel="Avg Daily Volume", grid_axis="x")

    # Segment type legend
    handles = [mpatches.Patch(color=c, label=t) for t, c in seg_type_colors.items()
               if t != "motorway"]
    fig.legend(handles=handles, title="Segment Type", loc="lower center",
               ncol=4, fontsize=8, title_fontsize=8,
               framealpha=0.15, facecolor=CREAM, edgecolor=MGRAY,
               bbox_to_anchor=(0.5, -0.04))

    plt.tight_layout(pad=1.5)
    out = os.path.join(OUTPUT_DIR, "fig4_top_segments.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=NAVY)
    plt.close()
    print(f"  ✓ Saved: {out}")

═══════════════════════════════════════════════════════════════════════════════
8. FIGURE 5 – PEAK-PERIOD PROFILE (AM/PM peaks)
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def fig_peak_profile(df, n_pairs=5):
    hourly = hourly_od_volumes(df)

    top_pairs = (
        hourly.groupby("od_pair")["od_vol"].sum()
        .nlargest(n_pairs).index.tolist()
    )
    hourly = hourly[hourly["od_pair"].isin(top_pairs)]

    hours_sorted = sorted(hourly["hour_code"].dropna().unique(), key=lambda x: int(x))

    hour_label_map = {}
    for hc in hours_sorted:
        h = int(hc)
        if h == 0: lbl = "12am"
        elif h < 12: lbl = f"{h}am"
        elif h == 12: lbl = "12pm"
        elif h == 24: lbl = "12am+"
        else: lbl = f"{h-12}pm"
        hour_label_map[hc] = lbl

    fig, ax = plt.subplots(figsize=(14, 5.5), facecolor=NAVY)
    fig.patch.set_facecolor(NAVY)
    ax.set_facecolor(CREAM)

    colors_cycle = [TEAL, AMBER, RUST, SAGE, "#7B5EA7"]

    for pair, color in zip(top_pairs, colors_cycle):
        sub = hourly[hourly["od_pair"] == pair].copy()
        sub = sub[sub["hour_code"].isin(hours_sorted)]
        sub["hour_idx"] = sub["hour_code"].map({h: i for i, h in enumerate(hours_sorted)})
        sub = sub.sort_values("hour_idx")

        ax.plot(sub["hour_idx"], sub["od_vol"], color=color, linewidth=2,
                marker="o", markersize=4, label=pair, alpha=0.9)
        ax.fill_between(sub["hour_idx"], sub["od_vol"], alpha=0.08, color=color)

    # Shade AM/PM peak windows
    am_peak = [i for i, h in enumerate(hours_sorted) if int(h) in range(7, 10)]
    pm_peak = [i for i, h in enumerate(hours_sorted) if int(h) in range(16, 20)]
    if am_peak:
        ax.axvspan(min(am_peak)-0.5, max(am_peak)+0.5, alpha=0.07, color=AMBER, label="AM Peak (7–9am)")
    if pm_peak:
        ax.axvspan(min(pm_peak)-0.5, max(pm_peak)+0.5, alpha=0.07, color=RUST, label="PM Peak (4–7pm)")

    ax.set_xticks(range(len(hours_sorted)))
    ax.set_xticklabels([hour_label_map[h] for h in hours_sorted],
                       rotation=45, ha="right", fontsize=8, color=NAVY)
    ax.set_ylabel("Avg Daily O-D Volume", fontsize=9, color=NAVY)
    style_ax(ax, title="Hourly Traffic Profile  ·  All Days  ·  Top OD Pairs",
             grid_axis="y")

    ax.legend(fontsize=8, loc="upper left", framealpha=0.85,
              facecolor=CREAM, edgecolor=LGRAY)

    plt.tight_layout(pad=1.5)
    out = os.path.join(OUTPUT_DIR, "fig5_peak_profile.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=NAVY)
    plt.close()
    print(f"  ✓ Saved: {out}")

═══════════════════════════════════════════════════════════════════════════════
9. CONSOLE SUMMARY TABLE
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def print_summary(top_routes):
    print("\n" + "═"*80)
    print("  STREETLIGHT TOP ROUTES · BI OD Analysis 2014404 · Kitsap Transit")
    print("  Jan–Dec 2025  |  All Days  |  All Day  |  All Vehicles CVD+")
    print("═"*80)
    print(f"  {'Rank':<5} {'OD Pair':<42} {'Avg Daily Vol':>14} {'Top Segment':<35}")
    print("  " + "─"*76)

    deduped = top_routes.drop_duplicates("od_pair").head(TOP_N).reset_index(drop=True)
    for i, row in deduped.iterrows():
        seg_short = row["segment"].split(" / ")[0][:33]
        print(f"  {i+1:<5} {row['od_pair']:<42} {int(row['od_vol']):>14,} {seg_short:<35}")

    print("═"*80 + "\n")

═══════════════════════════════════════════════════════════════════════════════
10. MAIN
═══════════════════════════════════════════════════════════════════════════════

In [ ]:
def main():
    csv_path = CSV_PATH

    # Allow override from command line: python script.py path/to/file.csv
    if len(sys.argv) > 1:
        csv_path = sys.argv[1]

    if not os.path.exists(csv_path):
        print(f"\n⚠️  File not found: {csv_path}")
        print("   Usage: python streetlight_top_routes.py [path/to/csv]")
        sys.exit(1)

    print("\n" + "═"*60)
    print("  StreetLight Top Routes Analyzer")
    print("═"*60)

    df = load_data(csv_path)
    top_routes = summarize_all_days(df)

    print_summary(top_routes)

    print("Generating figures...")
    fig_overview(top_routes)
    fig_hourly_heatmap(df)
    fig_day_of_week(df)
    fig_top_segments(df)
    fig_peak_profile(df)

    print(f"\n✅  All figures saved to: {OUTPUT_DIR}")
    print("  fig1_overview.png       – Ranked bar chart + summary table")
    print("  fig2_hourly_heatmap.png – Hour-by-hour heatmap for top pairs")
    print("  fig3_day_of_week.png    – Volume by day for top 6 pairs")
    print("  fig4_top_segments.png   – Street segments used per OD pair")
    print("  fig5_peak_profile.png   – AM/PM peak profile lines")
    print()

In [ ]:
if __name__ == "__main__":
    main()